# Inventory & Sales Analytics: pipeline walk-through
Production logic lives in `src/invsales/`; this notebook only *uses* it to explain the results. Data: the committed demo preset (simulated, see README).

In [1]:
import json
from pathlib import Path
import pandas as pd, plotly.express as px
from invsales import analytics, kpis
from invsales.repository import load_clean
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CLEAN = ROOT / 'data' / 'demo' / 'clean'
t = load_clean(CLEAN)
sales, products, inventory, restocks, warehouses = (t[k] for k in ('sales', 'products', 'inventory', 'restocks', 'warehouses'))
print({k: len(v) for k, v in t.items()})

{'warehouses': 3, 'products': 60, 'inventory': 180, 'restocks': 1863, 'sales': 103000}


## 1. What cleaning changed
The raw export was deliberately corrupted (duplicates, missing values, 3 date formats, `$` prices, inconsistent category text); `clean.py` fixes it and `validate.py` gates the load.

In [2]:
print(json.dumps(json.loads((CLEAN / 'cleaning_report.json').read_text()), indent=1))

{
 "rows_in": 105064,
 "unparseable_dates_dropped": 0,
 "duplicates_removed": 1246,
 "prices_filled_from_master": 1033,
 "invalid_rows_dropped": 818,
 "rows_out": 103000,
 "seconds_clean": 1.0,
 "rows": {
  "warehouses": 3,
  "products": 60,
  "sales": 103000,
  "inventory": 180,
  "restocks": 1863
 },
 "seconds_load": 2.6
}


## 2. Headline KPIs and trend

In [3]:
print(kpis.headline(sales, inventory))
px.line(kpis.monthly_sales(sales), x='month', y='revenue', markers=True, title='Monthly sales')

{'total_sales': 5358458.850000001, 'units_sold': 344322, 'total_stock': 26686, 'low_stock_items': 29}


## 3. Warehouses and categories

In [4]:
print(kpis.warehouse_summary(sales, inventory, warehouses).to_string(index=False))
kpis.category_summary(sales, products)

warehouse_id warehouse_name region     revenue  units  total_stock  skus  low_stock_items
         W01       Delhi DC  North  2078657.41 133546        10787    60                7
         W02      Mumbai DC   West  1915160.32 123273         8034    60               15
         W03     Chennai DC  South  1364641.12  87503         7865    60                7


,category,revenue,gross_profit,margin_pct,revenue_share_pct
0,Beverages,1476067.38,452491.28,30.7,27.5
1,Personal Care,1158362.72,341303.13,29.5,21.6
2,Snacks,1066321.62,309741.52,29.0,19.9
3,Staples,1008464.23,316998.73,31.4,18.8
4,Household,649242.9,201625.13,31.1,12.1


## 4. Fast and slow movers (stock turnover)
Turnover = units sold / average stock; slow movers tie up cash.

In [5]:
turn = kpis.stock_turnover(sales, inventory, restocks, products, level='product')
print(turn.nsmallest(5, 'stock_turnover')[['product_name', 'units_sold', 'avg_stock', 'stock_turnover', 'days_of_inventory']].to_string(index=False))
print(turn.nlargest(5, 'stock_turnover')[['product_name', 'units_sold', 'avg_stock', 'stock_turnover', 'days_of_inventory']].to_string(index=False))

       product_name  units_sold  avg_stock  stock_turnover  days_of_inventory
Salted Peanuts 200g        7599     1576.0            4.82              152.0
           Salt 1kg        7561     1565.0            4.83              151.0
   Soda Water 500ml        1103      208.5            5.29              138.0
         Wafers 75g        2285      404.5            5.65              129.0
       Toor Dal 1kg        7538     1317.5            5.72              128.0
         product_name  units_sold  avg_stock  stock_turnover  days_of_inventory
  Coconut Water 330ml        4416      226.5           19.50               37.0
Laundry Detergent 2kg        2239      132.5           16.90               43.0
      Granola Bar 6pk        4481      277.5           16.15               45.0
       Mango Drink 1L        7432      461.0           16.12               45.0
      Sunscreen 100ml        4497      281.5           15.98               46.0


## 5. ABC-XYZ segmentation

In [6]:
abc = analytics.abc_xyz(sales, products)
print(abc.groupby('segment').agg(products=('product_id', 'count'), revenue=('revenue', 'sum')).round(0))
px.scatter(abc, x='demand_cv', y='revenue', color='abc', hover_name='product_name', log_y=True)

         products    revenue
segment                     
AX             21  2966152.0
AY              7  1343405.0
BX             15   692719.0
BY              2    91571.0
CX             10   181230.0
CY              5    83382.0


## 6. Forecast accuracy (holdout of the last 30 days)
Model = recent level x weekday profile x yoy seasonal ratio, pooled per category; baseline = flat recent average.

In [7]:
groups = products.set_index('product_id').category
bt = analytics.backtest(sales, 30, groups)
print({k: v for k, v in bt.items() if k != 'per_product'})

{'horizon_days': 30, 'wape_model': 0.323, 'wape_naive': 0.3452, 'wape_weekly_model': 0.1393, 'wape_weekly_naive': 0.1585, 'bias_model': 0.0053}


## 7. Replenishment: what to order now

In [8]:
repl = analytics.replenishment(sales, inventory, products, warehouses)
due = repl[repl.below_rop].sort_values('suggested_order_cost', ascending=False)
print(f'{len(due)} of {len(repl)} warehouse-product pairs at/below the recommended reorder point; order value ${due.suggested_order_cost.sum():,.0f}')
print(f'{(repl.policy_gap > 0).mean():.0%} have a current reorder level below the recommended one')
due[['warehouse_name', 'product_name', 'current_stock', 'reorder_point', 'eoq', 'suggested_order_qty', 'suggested_order_cost']].head(10)

36 of 180 warehouse-product pairs at/below the recommended reorder point; order value $63,586
82% have a current reorder level below the recommended one


,warehouse_name,product_name,current_stock,reorder_point,eoq,suggested_order_qty,suggested_order_cost
65,Mumbai DC,Coffee Beans 500g,28,44.0,187.0,203,3749.41
18,Delhi DC,Granola Bar 6pk,14,30.0,160.0,176,3023.68
62,Mumbai DC,Green Tea 25 bags,28,41.0,153.0,166,2923.26
73,Mumbai DC,Salted Peanuts 200g,25,57.0,267.0,299,2822.56
93,Mumbai DC,Bleach 1L,14,18.0,157.0,161,2745.05
152,Chennai DC,Hand Wash 250ml,28,50.0,315.0,337,2618.49
166,Chennai DC,Hand Cream 75ml,0,13.0,113.0,126,2555.28
6,Delhi DC,Mango Drink 1L,38,43.0,282.0,287,2516.99
138,Chennai DC,Granola Bar 6pk,12,22.0,130.0,140,2405.20
137,Chennai DC,Namkeen Mix 400g,11,12.0,137.0,138,2097.60
